# Positional Encoding

তিনটি demo:

1. প্রমাণ যে শুধু self-attention নিজে permutation-invariant (একটি set operation) — input shuffle করে output আবার un-shuffle করলে প্রতিটি token-এর ফলাফল হুবহু একই ফিরে আসে।
2. Sinusoidal positional encoding: কাছাকাছি position-গুলো একই রকম "fingerprint" পায়, যা দূরত্বের সাথে মসৃণভাবে কমে যায়।
3. Relative-position rotation property: `PE(pos+k)` হলো `PE(pos)`-এর একটি স্থির linear (rotation) function, যা `pos`-এর উপর নির্ভর করে না।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান।

In [ ]:
import math
import numpy as np
import torch
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

## 1. Self-attention permutation-invariant

কোনো positional তথ্য ছাড়া একটি সাধারণ self-attention চালানো হয়, তারপর token-গুলো shuffle করে আবার চালিয়ে output un-shuffle করা হয়। দুই ফলাফলের পার্থক্য কার্যত শূন্য — অর্থাৎ attention input-কে একটি SET হিসেবে দেখে।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Self-attention permutation-invariant
# ---------------------------------------------------------------------------

def self_attention(X, Wq, Wk, Wv):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    d_k = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    weights = F.softmax(scores, dim=-1)
    return weights @ V


def permutation_invariance_demo():
    print("=" * 70)
    print("1. SELF-ATTENTION IS PERMUTATION-INVARIANT (NO POSITIONAL INFO)")
    print("=" * 70)

    T, d_model, d_k = 5, 8, 8
    X = torch.randn(T, d_model)
    Wq = torch.randn(d_model, d_k) * 0.3
    Wk = torch.randn(d_model, d_k) * 0.3
    Wv = torch.randn(d_model, d_k) * 0.3

    output_original = self_attention(X, Wq, Wk, Wv)

    # Token-গুলো shuffle করুন, আবার attention চালান, তারপর output-এ shuffle-টি
    # উল্টে দিন। Attention-এর যদি সত্যিই position-এর কোনো ধারণা না থাকে, তবে
    # এটি মূল, unshuffled run-এর হুবহু একই ফলাফল ফেরত দেবে।
    perm = torch.randperm(T)
    inverse_perm = torch.argsort(perm)
    X_shuffled = X[perm]
    output_shuffled = self_attention(X_shuffled, Wq, Wk, Wv)
    output_unshuffled = output_shuffled[inverse_perm]

    max_diff = (output_original - output_unshuffled).abs().max().item()
    print(f"Random permutation applied: {perm.tolist()}")
    print(f"Max difference between original output and un-shuffled-shuffled "
          f"output: {max_diff:.2e}")
    print("-> Effectively 0: attention treats the input as a SET. The model")
    print("   genuinely cannot tell 'A B C' from 'C A B' unless something else")
    print("   (positional encoding) tells it which token came from where.")


permutation_invariance_demo()

## 2. Sinusoidal positional encoding

README-এর সূত্র অনুযায়ী `(max_len, d_model)` আকারের sinusoidal PE matrix তৈরি করা হয়, তারপর একটি reference position (10)-এর সাথে বিভিন্ন position-এর cosine similarity মাপা হয় — দূরত্ব বাড়লে similarity সাধারণত কমে যায়।

In [ ]:
# ---------------------------------------------------------------------------
# 2. Sinusoidal positional encoding
# ---------------------------------------------------------------------------

def sinusoidal_positional_encoding(max_len, d_model):
    """একটি (max_len, d_model) matrix ফেরত দেয়, README-এর মতো PE(pos, dim)।"""
    positions = np.arange(max_len)[:, None]                      # (max_len, 1)
    dim_indices = np.arange(d_model)[None, :]                    # (1, d_model)
    angle_rates = 1.0 / (10000 ** ((2 * (dim_indices // 2)) / d_model))
    angles = positions * angle_rates                              # (max_len, d_model)

    pe = np.zeros((max_len, d_model))
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return pe


def cosine_similarity(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))


def sinusoidal_demo():
    print("\n" + "=" * 70)
    print("2. SINUSOIDAL POSITIONAL ENCODING: SIMILARITY DECAYS SMOOTHLY")
    print("=" * 70)

    d_model, max_len = 32, 50
    pe = sinusoidal_positional_encoding(max_len, d_model)

    reference_pos = 10
    print(f"cosine similarity between PE({reference_pos}) and PE(p), for various p:")
    for p in [10, 11, 12, 15, 20, 30, 49]:
        sim = cosine_similarity(pe[reference_pos], pe[p])
        distance = abs(p - reference_pos)
        print(f"  p={p:3d}  (distance {distance:3d})  cosine similarity = {sim:.4f}")

    print("\n-> Similarity is highest at distance 0 and generally decays as the")
    print("   position gets farther away -- each position gets a unique, but")
    print("   smoothly-varying, 'fingerprint' vector.")


sinusoidal_demo()

## 3. Relative-position rotation property

শুধু offset `k` থেকে একটি block-diagonal rotation matrix `M_k` তৈরি করা হয় এবং যাচাই করা হয় যে বিভিন্ন `pos`-এর জন্য `PE(pos+k) ≈ M_k @ PE(pos)`। এটিই সেই গাণিতিক ভিত্তি যার উপর পরে RoPE সরাসরি গড়ে উঠেছে।

In [ ]:
# ---------------------------------------------------------------------------
# 3. Relative-position rotation property
# ---------------------------------------------------------------------------

def rotation_matrix_for_offset(k, d_model):
    """Block-diagonal rotation matrix M_k তৈরি করে, যাতে তাত্ত্বিকভাবে প্রতিটি
    pos-এর জন্য PE(pos + k) = M_k @ PE(pos) হয় (README-এর derivation দেখুন)।"""
    M = np.zeros((d_model, d_model))
    for i in range(0, d_model, 2):
        theta = k / (10000 ** (i / d_model))
        cos_t, sin_t = np.cos(theta), np.sin(theta)
        # Angle-addition identity থেকে, v = [sin(pos*w), cos(pos*w)]-এর জন্য পাই:
        #   sin((pos+k)*w) = sin(pos*w)cos(k*w) + cos(pos*w)sin(k*w) =  cos_t*v[i] + sin_t*v[i+1]
        #   cos((pos+k)*w) = cos(pos*w)cos(k*w) - sin(pos*w)sin(k*w) = -sin_t*v[i] + cos_t*v[i+1]
        M[i, i] = cos_t
        M[i, i + 1] = sin_t
        M[i + 1, i] = -sin_t
        M[i + 1, i + 1] = cos_t
    return M


def relative_position_rotation_demo():
    print("\n" + "=" * 70)
    print("3. PE(pos + k) IS A FIXED LINEAR (ROTATION) FUNCTION OF PE(pos)")
    print("=" * 70)

    d_model, max_len = 16, 60
    pe = sinusoidal_positional_encoding(max_len, d_model)

    k = 5
    M_k = rotation_matrix_for_offset(k, d_model)

    print(f"Fixed offset k={k}. Checking PE(pos+{k}) ~= M_{k} @ PE(pos) for several pos:")
    print(f"{'pos':>5}  {'max abs error':>14}")
    for pos in [0, 3, 10, 25, 40]:
        predicted = M_k @ pe[pos]
        actual = pe[pos + k]
        max_error = np.abs(predicted - actual).max()
        print(f"{pos:>5}  {max_error:>14.2e}")

    print("\n-> The same rotation matrix M_k (built only from the offset k, not")
    print("   from pos) correctly maps PE(pos) to PE(pos+k) at every pos tested.")
    print("   This is the mathematical basis for the claim that a linear layer")
    print("   could, in principle, learn to attend by RELATIVE position using")
    print("   sinusoidal encodings -- the seed of the idea RoPE later builds on")
    print("   directly (Phase 03: Long-Context Techniques).")


relative_position_rotation_demo()

## সবগুলো demo একসাথে

`main()` উপরের সবগুলো demo এক সাথে ক্রমানুসারে চালায়। প্রতিটি demo ইতিমধ্যে নিজের cell-এ চলেছে, তাই দ্বিতীয়বার না চালানোর জন্য নিচে call-টি comment করা আছে।

In [ ]:
def main():
    permutation_invariance_demo()
    sinusoidal_demo()
    relative_position_rotation_demo()


# main()  # সব demo আবার একসাথে চালাতে uncomment করুন